# GSDensity base scores: simulated Visium-like slides (Fig. 1, Sup. Fig. 1)

Per-spot GSDensity relevance of the cluster_2 signature (20 genes) on each simulated slide, read by REV_Fig1 and
REV_SupFig1 from data/gsd_cache/sims/. GSDensity 0.1.3 runs in R (conda env gsdensity_env,
scripts/run_gsdensity.R) with the parameters of R4_F1_GSDensity (nn_use 300, restart 0.75, MCA dims 1:10,
seed 0), one MCA embedding per slide. Sup. Fig. 1: the 100 slides of 05_01_simulated_benchmarking (seeds 0-99);
Fig. 1: the slide of 05_00_simulated_dataset (seed 0).

In [1]:
import os
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

In [2]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import json

from sklearn.metrics import roc_auc_score, average_precision_score

In [3]:
sc.set_figure_params(frameon=False, fontsize=8)
sc.settings._vector_friendly = False
plt.rcParams["axes.grid"] = False

In [4]:
# set working directory
# repository root (the folder containing `notebooks/`); safe to re-run this cell
import sys
project_dir = os.getcwd()
if os.path.basename(project_dir) == "notebooks":
    project_dir = os.path.dirname(project_dir)
project_dir += "/"
sys.path.insert(0, project_dir + "notebooks")  # helpers.py and score_signatures.py
os.makedirs(project_dir + "logs", exist_ok=True)
working_dir = project_dir
os.chdir(working_dir)

# set figure directory
figure_dir = working_dir + "figures/"
os.makedirs(figure_dir, exist_ok=True)

# processed data directory
processed_data = project_dir + "processed_data/"

# source data directory
source_dir = working_dir + "source_data/"


# GSDensity cache of this dataset, its R task directories and the R logs
gsd_dir = working_dir + "data/gsd_cache/sims/"
task_root = gsd_dir + "_tasks/"
log_dir = working_dir + "logs/"

In [5]:
spot_size = 20

In [6]:
# 1, 3, 6. GSDensity bridge and the two simulators (helpers_parts/REV_GSD_base_sims.py)
from helpers import (
    export_gsdensity_task,
    run_gsdensity,
    read_gsdensity,
    gsdensity_signature_table,
    simulate_visium,
    simulate_visium_fig1,
)

In [7]:
# 1. GSDensity bridge: export a task, run scripts/run_gsdensity.R in gsdensity_env, read the results
gsd_rscript = working_dir + "scripts/run_gsdensity.R"

In [8]:
# 2. GSDensity parameters (R4_F1_GSDensity) and the signature name (05_00 / 05_01: signatures_dict = {"cluster_2": sig_genes})
GSD_PARAMS = {"nn_use": 300, "restart": 0.75, "dims": 10, "seed": 0, "spatial": True, "spatial_n": 10, "spatial_ntimes": 100}
SEEDS = list(range(100))
signature_name = "cluster_2"

In [9]:
# 4. Sup. Fig. 1: one GSDensity task per slide, run in 4 R sessions
slides = {}
for seed in SEEDS:
    adata, sig_genes, fold_changes, target_label = simulate_visium(seed=seed)
    task = export_gsdensity_task(adata, {signature_name: sig_genes}, task_root + f"SupFig1_seed{seed:03d}/", GSD_PARAMS)
    y = (adata.obs["cluster"] == target_label).values.astype(int)
    slides[seed] = {"task": task, "spots": adata.obs_names, "y": y, "target_label": target_label,
                    "genes": len(sig_genes)}
exit_codes = run_gsdensity([s["task"] for s in slides.values()], gsd_rscript, n_jobs=4, log=log_dir + "REV_GSD_base_sims_R.log")
print(f"R exit codes: {exit_codes}")

R exit codes: [0, 0, 0, 0]


In [10]:
# 5. Sup. Fig. 1: per-spot tables, index rows and per-slide performance
index_rows, summary_rows = [], []
for seed, s in slides.items():
    res = read_gsdensity(s["task"], s["spots"])
    tab = gsdensity_signature_table(res["scores"], res["labels"], signature_name, s["spots"])
    signature = f"SupFig1_seed{seed:03d}_{signature_name}"
    tab.to_csv(gsd_dir + f"{signature}.csv", index=False)
    positive = (tab["label"] == "positive").values
    index_rows.append({
        "dataset": "sims",
        "signature": signature,
        "name_in_notebook": signature_name,
        "used_in": "Sup. Fig. 1",
        "slide": f"seed {seed}",
        "generator": "05_01_simulated_benchmarking simulate_visium (default arguments)",
        "seed": seed,
        "target_label": s["target_label"],
        "genes_used": s["genes"],
        "n_spots": len(tab),
        "n_target_spots": s["y"].sum(),
        "n_positive_label": positive.sum(),
        "n_label_NA": tab["label"].isna().sum(),
        "n_zero_relevance": (tab["relevance"] == 0).sum(),
        "runtime_s": round(res["timing"]["total"], 1),
        "params": json.dumps(GSD_PARAMS),
        "file": f"data/gsd_cache/sims/{signature}.csv",
    })
    summary_rows.append({
        "seed": seed,
        "n_spots": len(tab),
        "prevalence": s["y"].mean(),
        "AUROC": roc_auc_score(s["y"], tab["relevance"].values),
        "AUPRC": average_precision_score(s["y"], tab["relevance"].values),
        "native label F1": 2 * (positive & (s["y"] == 1)).sum() / (positive.sum() + s["y"].sum()),
        "runtime_s": res["timing"]["total"],
    })
supfig1 = pd.DataFrame(summary_rows)
print(supfig1.describe().round(3).to_string())

          seed   n_spots  prevalence    AUROC    AUPRC  native label F1  runtime_s
count  100.000   100.000     100.000  100.000  100.000          100.000    100.000
mean    49.500  2913.990       0.254    0.747    0.500            0.244     20.448
std     29.011   760.952       0.092    0.093    0.197            0.181      5.803
min      0.000  1554.000       0.079    0.557    0.103            0.000     10.365
25%     24.750  2295.000       0.188    0.695    0.351            0.078     16.196
50%     49.500  3039.000       0.239    0.749    0.496            0.253     21.143
75%     74.250  3584.000       0.308    0.829    0.642            0.382     24.538
max     99.000  4012.000       0.534    0.931    0.890            0.627     34.889


In [11]:
# 7. Fig. 1: simulate the slide (the generator normalises once, as in the published panels) and run GSDensity
adata_fig1, sig_genes_fig1, fold_changes_fig1, target_label_fig1 = simulate_visium_fig1(seed=0)
y_fig1 = (adata_fig1.obs["cluster"] == target_label_fig1).values.astype(int)
print(f"Fig. 1 slide: {adata_fig1.n_obs} spots, {target_label_fig1} prevalence {y_fig1.mean():.3f}")
print(f"Signature: {sig_genes_fig1}")

task_fig1 = export_gsdensity_task(adata_fig1, {signature_name: sig_genes_fig1}, task_root + "Fig1_seed000/", GSD_PARAMS)
print(f"R exit codes: {run_gsdensity([task_fig1], gsd_rscript, log=log_dir + 'REV_GSD_base_sims_R_fig1.log')}")
res_fig1 = read_gsdensity(task_fig1, adata_fig1.obs_names)
print(res_fig1["timing"])
print(res_fig1["spatial_kld"].round(4).to_string())

Fig. 1 slide: 3705 spots, cluster_2 prevalence 0.203
Signature: ['GENE_01489', 'GENE_00579', 'GENE_01519', 'GENE_00535', 'GENE_00690', 'GENE_00602', 'GENE_01959', 'GENE_01853', 'GENE_02269', 'GENE_01318', 'GENE_00066', 'GENE_01523', 'GENE_00226', 'GENE_02417', 'GENE_02055', 'GENE_01899', 'GENE_01710', 'GENE_01136', 'GENE_00979', 'GENE_00686']


R exit codes: [0]
{'load': 6.5278, 'mca': 10.0362, 'nn': 4.0425, 'rwr': 3.769, 'spatial_kld': 0.8863, 'total': 25.7398}
           spatial.kld  rspatial.kld  rspatial.kld.sd  p.value  delta.kld  p.adj
cluster_2     -16.1182      -17.3089           0.2058      0.0     1.1907    0.0


In [12]:
# 8. Fig. 1: per-spot table, the dataset index and the summary
tab_fig1 = gsdensity_signature_table(res_fig1["scores"], res_fig1["labels"], signature_name, adata_fig1.obs_names)
signature = f"Fig1_seed000_{signature_name}"
tab_fig1.to_csv(gsd_dir + f"{signature}.csv", index=False)
positive = (tab_fig1["label"] == "positive").values
index_rows.append({
    "dataset": "sims",
    "signature": signature,
    "name_in_notebook": signature_name,
    "used_in": "Fig. 1",
    "slide": "seed 0 (Fig. 1 example)",
    "generator": "05_00_simulated_dataset simulate_visium (patchy domains)",
    "seed": 0,
    "target_label": target_label_fig1,
    "genes_used": len(sig_genes_fig1),
    "n_spots": len(tab_fig1),
    "n_target_spots": y_fig1.sum(),
    "n_positive_label": positive.sum(),
    "n_label_NA": tab_fig1["label"].isna().sum(),
    "n_zero_relevance": (tab_fig1["relevance"] == 0).sum(),
    "runtime_s": round(res_fig1["timing"]["total"], 1),
    "params": json.dumps(GSD_PARAMS),
    "file": f"data/gsd_cache/sims/{signature}.csv",
})
index = pd.DataFrame(index_rows)
index.to_csv(working_dir + "data/gsd_cache/INDEX_sims.csv", index=False)
print(index[["signature", "slide", "n_spots", "n_target_spots", "n_positive_label", "n_zero_relevance", "runtime_s"]]
      .tail(4).to_string(index=False))

fig1_row = {
    "seed": "Fig. 1",
    "n_spots": len(tab_fig1),
    "prevalence": y_fig1.mean(),
    "AUROC": roc_auc_score(y_fig1, tab_fig1["relevance"].values),
    "AUPRC": average_precision_score(y_fig1, tab_fig1["relevance"].values),
    "native label F1": 2 * (positive & (y_fig1 == 1)).sum() / (positive.sum() + y_fig1.sum()),
    "runtime_s": res_fig1["timing"]["total"],
}
summary = pd.concat([supfig1, pd.DataFrame([fig1_row])], ignore_index=True)
print("Sup. Fig. 1 slides (100 seeds):")
print(supfig1[["n_spots", "prevalence", "AUROC", "AUPRC", "native label F1", "runtime_s"]]
      .agg(["mean", "std", "min", "max"]).round(3).to_string())
print(f"\nFig. 1 slide: {pd.Series(fig1_row).to_string()}")

                signature                   slide  n_spots  n_target_spots  n_positive_label  n_zero_relevance  runtime_s
SupFig1_seed097_cluster_2                 seed 97     1840             289               928                 0       11.4
SupFig1_seed098_cluster_2                 seed 98     2295             367                 1                 0       14.2
SupFig1_seed099_cluster_2                 seed 99     4012            1513                12                77       25.6
   Fig1_seed000_cluster_2 seed 0 (Fig. 1 example)     3705             753               201                86       25.7
Sup. Fig. 1 slides (100 seeds):
       n_spots  prevalence  AUROC  AUPRC  native label F1  runtime_s
mean  2913.990       0.254  0.747  0.500            0.244     20.448
std    760.952       0.092  0.093  0.197            0.181      5.803
min   1554.000       0.079  0.557  0.103            0.000     10.365
max   4012.000       0.534  0.931  0.890            0.627     34.889

Fig. 1 slide